<h1 style="color:#1f77b4;">SARIMAX BANXICO API - EXAM</h1>

<h2 style="color: #6da7d1; border-bottom: 2px solid #6da7d1; padding-bottom: 5px;">Información del Proyecto</h2>

<table style="width: 100%; border-collapse: collapse;">
  <tr>
    <td style="padding: 10px; font-weight: bold; color: #6da7d1; width: 25%;">Materia:</td>
    <td style="padding: 10px;">Modelos No Lineales</td>
  </tr>
  <tr>
    <td style="padding: 10px; font-weight: bold; color: #6da7d1;">Profesor:</td>
    <td style="padding: 10px;">Pedro Martinez</td>
  </tr>
  <tr>
    <td style="padding: 10px; font-weight: bold; color: #6da7d1;">Fecha:</td>
    <td style="padding: 10px;">04 / 03 / 2026</td>
  </tr>
</table>

<h3 style="color: #6da7d1; margin-top: 20px;">Autoras:</h3>

* **Sarah Lucia Beltrán Gutiérrez**
* **Aissa Berenice**
* **Isabel Valladolid**
---

<h2 style="color: #6da7d1; border-bottom: 2px solid #6da7d1; padding-bottom: 5px;">Introducción y Objetivos</h2>

<p style="text-align: justify;">
En este proyecto se desarrolla un modelo <b style="color: #6da7d1;">SARIMAX</b> o <b style="color: #6da7d1;">SARIMA</b> para analizar y predecir el comportamiento del tipo de cambio <b>peso mexicano–dólar estadounidense (FIX)</b>, utilizando datos oficiales obtenidos mediante la API del <b>Banco de México</b>. 
</p>

<p style="text-align: justify;">
A partir de una serie temporal diaria, se busca evaluar la capacidad predictiva del modelo en el corto plazo, específicamente para el periodo del <b>5 al 13 de marzo de 2026</b>. El trabajo integra:
</p>

* **Descarga y procesamiento:** Automatización vía API de Banxico.
* **Justificación:** Selección de parámetros estacionales y exógenos.
* **Evaluación de desempeño:** Uso de la métrica <code style="color: #6da7d1;">MAPE</code> (Mean Absolute Percentage Error).

> <span style="color: #6da7d1;">**Propósito:**</span> Determinar la eficacia del modelo $SARIMA$ o $SARIMAX$ para anticipar movimientos del mercado cambiario en un entorno de alta volatilidad.


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import acf, pacf
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from dotenv import load_dotenv
import os
import requests


In [4]:
load_dotenv()
TOKEN = os.getenv("BANXICO_TOKEN") 
ID_SERIE = 'SF43718'    
FECHA_INICIO = '2020-01-01'
FECHA_FIN = '2026-03-04'

url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF43718/datos"
headers = {'Bmx-Token': TOKEN}

In [5]:
response = requests.get(url, headers=headers)
data = response.json()

In [6]:
# Extraer los datos y convertirlos a DataFrame
datos_serie = data['bmx']['series'][0]['datos']
df = pd.DataFrame(datos_serie)

# Limpieza y formato
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')
df['dato'] = df['dato'].astype(float)
df.set_index('fecha', inplace=True)

# Asignar frecuencia diaria y rellenar huecos (fines de semana/festivos)
df = df.asfreq('D')
df['dato'] = df['dato'].ffill()

df.head()

,dato
fecha,
1991-11-12,3.0735
1991-11-13,3.0712
1991-11-14,3.0718
1991-11-15,3.0684
1991-11-16,3.0684


<h2 style="color: #6da7d1; border-bottom: 2px solid #6da7d1; padding-bottom: 5px;">Análisis Exploratorio y Estacionariedad</h2>

<p style="text-align: justify;">
Se ha completado satisfactoriamente la <b>carga del dataset</b> desde la API de Banxico. Se realizaron los ajustes necesarios en los <b>índices temporales</b>, asegurando la continuidad de la serie y el tratamiento adecuado de fechas no hábiles para el tipo de cambio FIX.

Los datos están normalizados y listos para la etapa de modelado econométrico.</p>


<h3 style="color: #6da7d1; margin-top: 25px;">Siguiente Paso: Identificación de Parámetros</h3>

<p style="text-align: justify;">
A continuación, procederemos con la búsqueda exhaustiva de los hiperparámetros óptimos para el modelo <b style="color: #6da7d1;">SARIMAX</b> y <b style="color: #6da7d1;">SARIMA</b>. Esto incluye:
</p>

* **Componente no estacional:** Identificación de $(p, d, q)$.
* **Componente estacional:** Determinación de $(P, D, Q)_S$.
* **Criterio de Selección:** Evaluación mediante el **AIC** (Akaike Information Criterion) para encontrar el equilibrio entre ajuste y parsimonia.

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['dato'].index, y=df['dato'].values, mode='lines', name='Carreras Diarias'))

# Resaltar la diferencia entre Lunes/Jueves y Fines de Semana
fig.update_layout(
    title='Volumen Diario de Carreras en la MLB',
    xaxis_title='Fecha',
    yaxis_title='Total Carreras (Runs)'
)
fig.show()

Como no nos sirve tomar la serie desde 1995, ya que buscamos predecir solo 9 dias adelante, tomaremos de una fecha mas cercana como 2023-2025

In [8]:
df = df[df.index >= '2023-01-01']
df.head()

,dato
fecha,
2023-01-01,19.4715
2023-01-02,19.4883
2023-01-03,19.4220
2023-01-04,19.3568
2023-01-05,19.3672


In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['dato'].index, y=df['dato'].values, mode='lines', name='Carreras Diarias'))

# Resaltar la diferencia entre Lunes/Jueves y Fines de Semana
fig.update_layout(
    title='Volumen Diario de Carreras en la MLB',
    xaxis_title='Fecha',
    yaxis_title='Total Carreras (Runs)'
)
fig.show()

In [10]:
# @title Realizamos pruebas de estacionareidad

def check_stationarity(series, title="Serie Original"):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'Estadístico ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"¿Es estacionaria? {'SÍ' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. Revisamos la serie original
check_stationarity(df['dato'], "Nivel Original")

# 2. Aplicamos Primera Diferencia (d=1)
df_diff = df['dato'].diff()

# 3. Revisamos la serie diferenciada
check_stationarity(df_diff, "Primera Diferencia (d=1)")

# Creamos una figura con 2 columnas (Subplots)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Serie Original (No Estacionaria)", "Serie Diferenciada (Estacionaria d=1)")
)

# Gráfico 1: Serie Original
fig.add_trace(
    go.Scatter(x=df['dato'].index, y=df['dato'].values, name='Original'),
    row=1, col=1
)

# Gráfico 2: Serie Diferenciada
fig.add_trace(
    go.Scatter(x=df_diff.index, y=df_diff.values, name='Diferenciada'),
    row=1, col=2
)

# Ajustes de diseño
fig.update_layout(
    title_text="Comparativa: Efecto de la Diferenciación",
    showlegend=False, # Ocultamos leyenda
    height=500
)

fig.show()

ADF Test: Nivel Original
Estadístico ADF: -1.5527
p-value: 0.5073
¿Es estacionaria? NO

ADF Test: Primera Diferencia (d=1)
Estadístico ADF: -35.6926
p-value: 0.0000
¿Es estacionaria? SÍ



In [11]:
df_analysis = df_diff.dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(df_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(df_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(df_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

# Resaltar lags estacionales (7, 14, 21, 28) con líneas verticales rojas tenues
for i in [7, 14, 21, 28]:
    fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()

In [12]:
TEST_DAYS = 21

train = df['dato'].iloc[:-TEST_DAYS]
test = df['dato'].iloc[-TEST_DAYS:]

# Con utilizar una diferenciacion es suficiente, tu decides cual utilizar
# (revisa la forma canónica) para verificar que el modelo despues de simplificar
# ya se trata de una serie estacionaria
model = SARIMAX(train,
                order=(3, 0, 1),
                seasonal_order=(0, 1, 1, 7))

results = model.fit(disp=False)

# Predecimos n pasos hacia el futuro (donde n = tamaño del test)
forecast_object = results.get_forecast(steps=len(test))
forecast_vals = forecast_object.predicted_mean
conf_int = forecast_object.conf_int(alpha=0.05) # Intervalo del 95%

# Metricas de error
rmse = np.sqrt(mean_squared_error(test, forecast_vals))
# Ojo con el MAPE, hay algunos días con 0 carreras y esto da valores muy grandes
mape = mean_absolute_percentage_error(test, forecast_vals)
mae = mean_absolute_error(test, forecast_vals)

print(f"\n--- Errores del modelo ---")
print(f"RMSE: {rmse:.2f} Carreras")
print(f"MAPE: {mape:.2%}")
print(f"MAE: {mae:.2f} Carreras")


print(results.summary())

# Grafica
fig = go.Figure()

# Train
fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines',
    name='Train',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

# Test
fig.add_trace(go.Scatter(
    x=test.index, y=test,
    name='Test',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6)
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=forecast_vals,
    name='SARIMA',
    line=dict(color='#ff7f0e', width=3, dash='dot')
))

# Intervalos de Confianza
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 0],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(255, 127, 14, 0.2)',
    name='Int. Confianza 95%', hoverinfo='skip'
))

# Titulos
fig.update_layout(
    title=f'<b>Modelo SARIMA: Pronóstico de Carreras MLB</b><br><sup>',
    xaxis_title='Fecha',
    yaxis_title='Total de Carreras',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()

c:\Users\sarah\apps\sem_6\ModNoLin\SARIMAX-API-Banxico\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals




--- Errores del modelo ---
RMSE: 0.13 Carreras
MAPE: 0.42%
MAE: 0.07 Carreras
                                     SARIMAX Results                                     
Dep. Variable:                              dato   No. Observations:                 1138
Model:             SARIMAX(3, 0, 1)x(0, 1, 1, 7)   Log Likelihood                 905.055
Date:                           Wed, 04 Mar 2026   AIC                          -1798.109
Time:                                   18:54:25   BIC                          -1767.924
Sample:                               01-01-2023   HQIC                         -1786.706
                                    - 02-11-2026                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0593      0.379      0.156

In [13]:
# 1. Definimos el rango de fechas solicitado
fecha_inicio = '2026-03-05'
fecha_fin = '2026-03-13'

# 2. Generamos el pronóstico fuera de muestra (out-of-sample)
# Usamos el objeto 'results' que ya tienes entrenado en tu celda anterior
pronostico_entrega = results.get_prediction(start=pd.to_datetime(fecha_inicio), 
                                            end=pd.to_datetime(fecha_fin))

# 3. Extraemos los valores predichos (la media)
df_entrega = pronostico_entrega.predicted_mean.to_frame(name='Prediccion_Tipo_Cambio_FIX')

# 4. Ajustes de formato para el Excel
df_entrega.index.name = 'Fecha'
df_entrega = df_entrega.reset_index()

# 5. Exportar a Excel
nombre_archivo = "Predicciones_5-13_Mar_2026.xlsx"
df_entrega.to_excel(nombre_archivo, index=False)

print(f"Archivo '{nombre_archivo}' generado con éxito.")
display(df_entrega)

Archivo 'Predicciones_5-13_Mar_2026.xlsx' generado con éxito.


,Fecha,Prediccion_Tipo_Cambio_FIX
0,2026-03-05,17.263778
1,2026-03-06,17.244860
2,2026-03-07,17.248953
3,2026-03-08,17.252982
4,2026-03-09,17.266499
5,2026-03-10,17.282514
6,2026-03-11,17.273079
7,2026-03-12,17.277803
8,2026-03-13,17.258904
